# Step 8 — DeBERTa-v3-large fine-tune (different architecture, real diversity)

**Goal:** add a 4th anchor that is genuinely uncorrelated with SciNCL/SPECTER2 (which are both BERT-base trained on scientific corpora and correlate ~0.94 with each other). DeBERTa-v3 uses **disentangled attention** and was trained on a different pre-training corpus, so its representation should diverge from BERT-class scientific encoders. Expected correlation ~0.80-0.88 (vs the 0.94+ we keep seeing).

**Why DeBERTa-v3-large beats the BERT-class candidates we already tried:**
- Architecture: DeBERTa-v3 disentangles content and position attention (different signal).
- Size: 435M params (vs SPECTER2/SciNCL's 110M). More capacity for niche domain features.
- Pre-training: ELECTRA-style + 130GB CC + corpus, generic but high-quality text understanding.
- Track record: top of GLUE/SuperGLUE for text classification.

**Recipe (tuned for DeBERTa-v3-large stability — different from BERT-base step 6):**
- Model: `microsoft/deberta-v3-large` loaded with `torch_dtype=fp32`
  - The Hub `model.safetensors` is fp16; without the override `transformers>=4.41` puts the encoder in fp16 and head in fp32 -> dtype-mismatch RuntimeError.
- Input: `title [SEP] abstract`, max_len 256
- Head: `Linear(1024 -> 1)`, dropout 0.1
- Loss: `SmoothL1Loss(beta=1.0)`
- Optimizer: AdamW, encoder LR `2e-5`, head LR `1e-4`, weight_decay `0.01`
  - First attempt with `LR_HEAD=1e-3` (the BERT-base default) diverged to NaN. Lowered to `1e-4`.
  - First attempt with `LR_ENCODER=1e-5` was stable but under-converged at 5 epochs (per-fold ~0.50). Bumped to `2e-5`.
- Schedule: 15% warmup + linear decay, **pure fp32** (DeBERTa-v3's disentangled attention overflows in fp16/bf16), grad clip 1.0
- 5 folds × 3 seeds [252, 253, 254] = 15 models, **6 epochs** per fold
- `BATCH_TRAIN = 4`, `GRAD_ACCUM_STEPS = 4` (effective batch = 16, fits fp32 deberta-large in A100 40GB)
- Per-fold best epoch by validation round-QWK
- Constrained threshold tuner (lambda=0.5)

**Time budget on A100:** ~12-15 min per fold × 15 folds ≈ 3-4 hours total (fp32 + 6 epochs).

## 1. GPU + dependencies

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q --upgrade "transformers>=4.41" "accelerate>=0.30" "sentencepiece>=0.2" scikit-learn pandas "numpy<2" scipy

## 2. Data setup (Colab / Kaggle / local)

The next cell auto-detects the platform and locates `asp_data*.zip` (containing `train.csv`, `public_test.csv`, `private_test.csv`, `Test_Submission.csv`, and `abstracts_merged_v2.csv` or `_v3.csv`).

Source priority:
1. `ASP_DATA_ZIP=/path/to/asp_data.zip` env var (explicit override).
2. `asp_data*.zip` in cwd or any parent directory up to the repo root (local checkout — `asp_data_v3.zip` is at the repo root).
3. Kaggle: any `*.zip` under `/kaggle/input/` (attach the dataset to the kernel).
4. Colab: interactive upload widget (only triggered if no zip was found).

Working directory:
- Colab → `/content/work`
- Kaggle → `/kaggle/working/asp_work`
- Local → `./work` next to the notebook

In [ ]:
import os, pathlib, zipfile, shutil

# --- Platform-agnostic data setup ----------------------------------------
# Works on Colab, Kaggle, and local Jupyter.
# Looks for a zip containing train.csv / *_test.csv / abstracts_merged_v*.csv.
# Source priority:
#   1. ASP_DATA_ZIP env var (explicit override)
#   2. asp_data*.zip in cwd (local checkout typically has this)
#   3. Kaggle dataset paths (/kaggle/input/**/*.zip)
#   4. Colab interactive upload via google.colab.files

def _detect_platform():
    if 'COLAB_RELEASE_TAG' in os.environ or 'COLAB_GPU' in os.environ:
        return 'colab'
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or pathlib.Path('/kaggle/working').exists():
        return 'kaggle'
    return 'local'

PLATFORM = _detect_platform()
print('platform =', PLATFORM)

# Pick a writable working directory per platform.
if PLATFORM == 'colab':
    WORK = pathlib.Path('/content/work')
elif PLATFORM == 'kaggle':
    WORK = pathlib.Path('/kaggle/working/asp_work')
else:
    WORK = pathlib.Path.cwd() / 'work'

DATA = WORK / 'data'
OUT = WORK / 'outputs'
RUN_DIR = OUT / 'deberta_v3_finetune'
for d in [WORK, DATA, OUT, RUN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def _find_local_zip():
    env_zip = os.environ.get('ASP_DATA_ZIP')
    if env_zip and pathlib.Path(env_zip).exists():
        return pathlib.Path(env_zip)
    # cwd + parents — handy when notebook is in repo/notebooks/
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        for pat in ('asp_data*.zip', 'asp_data.zip'):
            for cand in sorted(base.glob(pat)):
                return cand
        if (base / '.git').exists():
            break  # don't escape the repo
    return None

def _find_kaggle_zip():
    root = pathlib.Path('/kaggle/input')
    if not root.exists():
        return None
    for cand in sorted(root.rglob('asp_data*.zip')):
        return cand
    # fall back: any zip in /kaggle/input
    for cand in sorted(root.rglob('*.zip')):
        return cand
    return None

def _ingest_zip(zip_path):
    print('using zip:', zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(DATA)
    print('extracted ->', DATA)

# Decide source.
zip_path = _find_local_zip()
if zip_path is None and PLATFORM == 'kaggle':
    zip_path = _find_kaggle_zip()

if zip_path is not None:
    _ingest_zip(zip_path)
elif PLATFORM == 'colab':
    from google.colab import files
    print('No local zip found; falling back to Colab upload widget.')
    uploaded = files.upload()
    for name, content in uploaded.items():
        target = WORK / name
        target.write_bytes(content)
        if name.lower().endswith('.zip'):
            _ingest_zip(target)
else:
    raise FileNotFoundError(
        'No data zip found. Set ASP_DATA_ZIP=/path/to/asp_data.zip, place '
        'asp_data*.zip in cwd or a parent, or attach the dataset on Kaggle.')

print('\nFiles in DATA:')
for p in sorted(DATA.glob('*')):
    print(' ', p.name, p.stat().st_size)

## 3. Load + merge

In [ ]:
import pandas as pd, numpy as np, json, re, time
from pathlib import Path

# DATA / RUN_DIR were defined in the platform-agnostic setup cell.
# This re-asserts them as Path so the cell is safe to re-run on its own.
DATA = Path(DATA)
RUN_DIR = Path(RUN_DIR)

train = pd.read_csv(DATA / 'train.csv')
public = pd.read_csv(DATA / 'public_test.csv')
private = pd.read_csv(DATA / 'private_test.csv')
sample = pd.read_csv(DATA / 'Test_Submission.csv')
abstracts_file = DATA / 'abstracts_merged_v2.csv'
if not abstracts_file.exists():
    abstracts_file = DATA / 'abstracts_merged_v3.csv'
abstracts = pd.read_csv(abstracts_file)
print('using abstract cache:', abstracts_file.name)

abs_map = abstracts[['source_split', 'id', 'abstract', 'has_abstract']]

def attach(df, split):
    df = df.copy()
    df['source_split'] = split
    out = df.merge(abs_map, on=['source_split', 'id'], how='left')
    out['abstract'] = out['abstract'].fillna('')
    out['has_abstract'] = out['has_abstract'].fillna(False).astype(bool)
    return out

train_full = attach(train, 'train').reset_index(drop=True)
public_full = attach(public, 'public_test').reset_index(drop=True)
private_full = attach(private, 'private_test').reset_index(drop=True)
for name, df in [('train', train_full), ('public', public_full), ('private', private_full)]:
    print(f'{name}: rows={len(df)}, has_abstract={int(df["has_abstract"].sum())} ({df["has_abstract"].mean():.1%})')

## 4. Tokenizer + dataset

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = 'microsoft/deberta-v3-large'
MAX_LEN = 256
BATCH_TRAIN = 4        # fp32 deberta-v3-large peak ~30GB on A100; smaller batch + more accum
BATCH_EVAL = 16
GRAD_ACCUM_STEPS = 4   # effective batch = 16 (matches step 3b/6)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
SEP = tokenizer.sep_token

def build_input(title, abstract):
    title = '' if pd.isna(title) else str(title).strip()
    abstract = '' if pd.isna(abstract) else str(abstract).strip()
    return f'{title}{SEP}{abstract}' if abstract else title

class PaperDataset(Dataset):
    def __init__(self, df, with_label):
        self.texts = [build_input(t, a) for t, a in zip(df['title'], df['abstract'])]
        self.labels = df['Label'].astype(np.float32).to_numpy() if with_label else None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = {'text': self.texts[idx], 'idx': idx}
        if self.labels is not None:
            item['label'] = self.labels[idx]
        return item

def collate(batch):
    texts = [b['text'] for b in batch]
    enc = tokenizer(texts, padding=True, truncation=True, max_length=MAX_LEN,
                    return_tensors='pt', return_token_type_ids=False)
    out = {'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask'],
           'idx': torch.tensor([b['idx'] for b in batch], dtype=torch.long)}
    if 'label' in batch[0]:
        out['label'] = torch.tensor([b['label'] for b in batch], dtype=torch.float32)
    return out

print('tokenizer loaded:', MODEL_NAME, '; SEP =', SEP)

## 5. Model

In [ ]:
import torch.nn as nn

class DeBERTaRegressor(nn.Module):
    def __init__(self, model_name=MODEL_NAME, dropout=0.1):
        super().__init__()
        # IMPORTANT: deberta-v3-large's model.safetensors on the Hub is stored in fp16.
        # transformers>=4.41 honours the on-disk dtype, so without an explicit override
        # the encoder loads in fp16 while our head Linear is fp32 -> RuntimeError
        # ("mat1 and mat2 must have the same dtype, but got Half and Float").
        # Force fp32 here. fp32 is also required because deberta-v3 disentangled
        # attention overflows in fp16/bf16 (NaN loss).
        try:
            self.encoder = AutoModel.from_pretrained(model_name, torch_dtype=torch.float32)
        except TypeError:
            # transformers >= 5.0 renamed torch_dtype -> dtype
            self.encoder = AutoModel.from_pretrained(model_name, dtype=torch.float32)
        self.encoder = self.encoder.float()
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(self.encoder.config.hidden_size, 1)
        nn.init.trunc_normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Mean pooling with attention mask.
        # DeBERTa-v3 was pre-trained with Replaced Token Detection (ELECTRA-style).
        # Unlike BERT/SPECTER2 (NSP/contrastive on [CLS]), the position-0 token is NOT
        # a sequence summary, so CLS pooling under-performs. Masked mean over all
        # tokens is the canonical pooling for deberta-v3 fine-tuning (FB3, ELL, etc.).
        last_hidden = outputs.last_hidden_state                           # (B, T, H)
        mask = attention_mask.unsqueeze(-1).float()                       # (B, T, 1)
        summed = (last_hidden * mask).sum(dim=1)                          # (B, H)
        counts = mask.sum(dim=1).clamp(min=1.0)                           # (B, 1)
        pooled = summed / counts                                          # (B, H)
        return self.head(self.dropout(pooled)).squeeze(-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)

## 6. Single-fold training (with gradient accumulation)

In [ ]:
from sklearn.metrics import cohen_kappa_score
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

EPOCHS = 5             # mean-pool run peaked at epoch 5/6 (0.57) and regressed at 6 (0.55)
LR_ENCODER = 2e-5      # top-layer LR; lower layers get scaled down by LLRD below
LR_HEAD = 1e-4         # IMPORTANT: 1e-3 (BERT-base default) blows up deberta-large -> NaN
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.15
GRAD_CLIP = 1.0
LLRD_DECAY = 0.95      # layer-wise LR decay; standard DeBERTa-v3 fine-tune trick
                       # (FB3, ELL) — lower layers learn slower to preserve generic
                       # representations, top layers adapt freely to the task.

USE_AMP = False        # DeBERTa-v3 disentangled attention NaNs in fp16/bf16

@torch.no_grad()
def predict(model, loader):
    model.eval()
    n = len(loader.dataset)
    out = np.zeros(n, dtype=np.float32)
    for batch in loader:
        ids = batch['input_ids'].to(device, non_blocking=True)
        mask = batch['attention_mask'].to(device, non_blocking=True)
        preds = model(ids, mask).float().cpu().numpy()
        idx = batch['idx'].numpy()
        out[idx] = preds
    if not np.isfinite(out).all():
        n_bad = int((~np.isfinite(out)).sum())
        print(f'  WARNING: predict() produced {n_bad} non-finite scores; clamping to 3.0')
        out = np.where(np.isfinite(out), out, 3.0)
    return np.clip(out, 1.0, 5.0)


def build_llrd_param_groups(model, lr_encoder, lr_head, weight_decay, decay):
    """Layer-wise LR decay for DeBERTa-v3 (or any HF encoder).

    Each transformer layer i (0 = closest to embeddings, N-1 = closest to head)
    gets LR = lr_encoder * decay^(N-1-i). Embeddings get the smallest LR.
    Bias and LayerNorm parameters get weight_decay = 0.
    """
    no_decay = ('bias', 'LayerNorm.weight', 'LayerNorm.bias')
    encoder = model.encoder
    n_layers = len(encoder.encoder.layer)
    groups = []

    def _add(named_params, lr):
        decay_p = [p for n, p in named_params if not any(nd in n for nd in no_decay)]
        nodecay_p = [p for n, p in named_params if any(nd in n for nd in no_decay)]
        if decay_p:
            groups.append({'params': decay_p, 'lr': lr, 'weight_decay': weight_decay})
        if nodecay_p:
            groups.append({'params': nodecay_p, 'lr': lr, 'weight_decay': 0.0})

    # Embeddings (incl. relative position embeddings) — lowest LR.
    emb_lr = lr_encoder * (decay ** (n_layers + 1))
    _add(list(encoder.embeddings.named_parameters()), emb_lr)
    if hasattr(encoder.encoder, 'rel_embeddings'):
        _add([('rel_embeddings.weight', encoder.encoder.rel_embeddings.weight)], emb_lr)
    if hasattr(encoder.encoder, 'LayerNorm'):
        _add([('encoder.LayerNorm.weight', encoder.encoder.LayerNorm.weight),
              ('encoder.LayerNorm.bias', encoder.encoder.LayerNorm.bias)], emb_lr)

    # Transformer layers — LR decays by `decay` per layer downward.
    for i, layer in enumerate(encoder.encoder.layer):
        layer_lr = lr_encoder * (decay ** (n_layers - 1 - i))
        _add(list(layer.named_parameters()), layer_lr)

    # Regression head — much higher LR.
    _add(list(model.head.named_parameters()), lr_head)
    return groups


def train_one_fold(train_df, valid_df, public_df, private_df, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

    train_loader = DataLoader(PaperDataset(train_df, with_label=True),
                              batch_size=BATCH_TRAIN, shuffle=True,
                              collate_fn=collate, num_workers=2, pin_memory=True)
    valid_loader = DataLoader(PaperDataset(valid_df, with_label=True),
                              batch_size=BATCH_EVAL, shuffle=False,
                              collate_fn=collate, num_workers=2, pin_memory=True)
    public_loader = DataLoader(PaperDataset(public_df.assign(Label=0), with_label=False),
                               batch_size=BATCH_EVAL, shuffle=False,
                               collate_fn=collate, num_workers=2, pin_memory=True)
    private_loader = DataLoader(PaperDataset(private_df.assign(Label=0), with_label=False),
                                batch_size=BATCH_EVAL, shuffle=False,
                                collate_fn=collate, num_workers=2, pin_memory=True)

    model = DeBERTaRegressor().to(device)
    param_groups = build_llrd_param_groups(model, LR_ENCODER, LR_HEAD, WEIGHT_DECAY, LLRD_DECAY)
    optim = AdamW(param_groups)
    total_steps = (EPOCHS * len(train_loader)) // GRAD_ACCUM_STEPS
    scheduler = get_linear_schedule_with_warmup(
        optim, num_warmup_steps=int(WARMUP_RATIO * total_steps),
        num_training_steps=total_steps)
    loss_fn = nn.SmoothL1Loss(beta=1.0)

    best_state = None
    best_qwk = -1.0
    for epoch in range(EPOCHS):
        model.train()
        running = 0.0
        n_skipped = 0
        t0 = time.time()
        optim.zero_grad(set_to_none=True)
        for step, batch in enumerate(train_loader):
            ids = batch['input_ids'].to(device, non_blocking=True)
            mask = batch['attention_mask'].to(device, non_blocking=True)
            y = batch['label'].to(device, non_blocking=True)
            preds = model(ids, mask)
            loss = loss_fn(preds, y) / GRAD_ACCUM_STEPS
            if not torch.isfinite(loss):
                n_skipped += 1
                optim.zero_grad(set_to_none=True)
                continue
            loss.backward()
            if (step + 1) % GRAD_ACCUM_STEPS == 0:
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optim.step()
                scheduler.step()
                optim.zero_grad(set_to_none=True)
            running += loss.item() * GRAD_ACCUM_STEPS
        val_scores = predict(model, valid_loader)
        rounded = np.clip(np.round(val_scores), 1, 5).astype(int)
        qwk = cohen_kappa_score(valid_df['Label'].astype(int).to_numpy(), rounded, weights='quadratic')
        skipped_msg = f'  skipped={n_skipped}' if n_skipped else ''
        print(f'  epoch {epoch+1}/{EPOCHS}  loss={running/max(len(train_loader),1):.3f}  val_round_QWK={qwk:.4f}  ({time.time()-t0:.1f}s){skipped_msg}')
        if qwk > best_qwk:
            best_qwk = qwk
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is None:
        raise RuntimeError('Training produced no usable epoch (all NaN). Check LR / precision.')
    model.load_state_dict(best_state)
    return predict(model, valid_loader), predict(model, public_loader), predict(model, private_loader), best_qwk

print(f'train_one_fold ready (fp32, mean-pool, LLRD={LLRD_DECAY}, LR_top={LR_ENCODER}, EPOCHS={EPOCHS})')

## 7. Repeated CV (5 folds x 3 seeds)

In [ ]:
from sklearn.model_selection import StratifiedKFold

FOLDS = 5
SEEDS = [252, 253, 254]

y_class = train_full['Label'].astype(int).to_numpy()
oof_sum = np.zeros(len(train_full), dtype=np.float64)
oof_count = np.zeros(len(train_full), dtype=np.float64)
public_sum = np.zeros(len(public_full), dtype=np.float64)
private_sum = np.zeros(len(private_full), dtype=np.float64)
n_models = 0
fold_log = []

for seed in SEEDS:
    cv = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=seed)
    for fold, (tr_idx, va_idx) in enumerate(cv.split(train_full, y_class), start=1):
        print(f'\n=== seed={seed} fold={fold}/{FOLDS} ===')
        t0 = time.time()
        train_df = train_full.iloc[tr_idx].reset_index(drop=True)
        valid_df = train_full.iloc[va_idx].reset_index(drop=True)
        val_scores, pub_scores, priv_scores, best_qwk = train_one_fold(
            train_df, valid_df, public_full, private_full, seed * 1000 + fold)
        oof_sum[va_idx] += val_scores
        oof_count[va_idx] += 1.0
        public_sum += pub_scores
        private_sum += priv_scores
        n_models += 1
        fold_log.append({'seed': seed, 'fold': fold, 'best_round_qwk': float(best_qwk),
                         'minutes': round((time.time()-t0)/60, 2)})

print('\n=== Done. trained', n_models, 'models. ===')
oof_scores = oof_sum / np.clip(oof_count, 1.0, None)
public_scores = public_sum / n_models
private_scores = private_sum / n_models
print('OOF round-QWK:',
      cohen_kappa_score(y_class, np.clip(np.round(oof_scores), 1, 5).astype(int), weights='quadratic'))

## 8. Constrained threshold tuning

In [ ]:
from scipy.optimize import differential_evolution
from sklearn.metrics import f1_score, mean_absolute_error

TRAIN_DIST = pd.Series(y_class).value_counts(normalize=True).reindex([1, 2, 3, 4, 5], fill_value=0).to_numpy()
DIST_PENALTY_LAMBDA = 0.5

def scores_to_labels(scores, thresholds):
    return np.digitize(scores, np.sort(np.asarray(thresholds, dtype=float))) + 1

def predicted_dist(labels):
    return pd.Series(labels).value_counts(normalize=True).reindex([1, 2, 3, 4, 5], fill_value=0).to_numpy()

def tune_thresholds_constrained(y_true, oof, lambd=DIST_PENALTY_LAMBDA, seed=42):
    def objective(raw):
        thr = np.sort(raw)
        gap = np.min(np.diff(thr))
        gap_pen = 0.0 if gap >= 0.03 else (0.03 - gap) * 5.0
        labels = scores_to_labels(oof, thr)
        qwk = cohen_kappa_score(y_true, labels, weights='quadratic')
        dist_pen = float(np.sum(np.abs(predicted_dist(labels) - TRAIN_DIST)))
        return -qwk + gap_pen + lambd * dist_pen
    bounds = [(1.4, 2.5), (1.8, 2.9), (2.2, 3.4), (2.6, 4.2)]
    res = differential_evolution(objective, bounds, seed=seed, maxiter=120, popsize=15,
                                 polish=True, updating='immediate', workers=1)
    thr = np.sort(res.x)
    return thr, cohen_kappa_score(y_true, scores_to_labels(oof, thr), weights='quadratic')

thresholds, oof_qwk = tune_thresholds_constrained(y_class, oof_scores)
oof_pred = scores_to_labels(oof_scores, thresholds)
print('Constrained-tuned OOF QWK =', round(oof_qwk, 4))
print('  (compare to SPECTER2 0.6373, SciNCL 0.6269, SciBERT 0.6360)')
print('thresholds =', thresholds.tolist())
print('OOF predicted dist =', dict(zip([1,2,3,4,5], predicted_dist(oof_pred).round(3).tolist())))
print('TRAIN actual dist  =', dict(zip([1,2,3,4,5], TRAIN_DIST.round(3).tolist())))
print('OOF MAE =', round(mean_absolute_error(y_class, oof_pred), 4))
print('OOF macro-F1 =', round(f1_score(y_class, oof_pred, average='macro'), 4))

## 9. Save artefacts

In [ ]:
public_pred = scores_to_labels(public_scores, thresholds)
private_pred = scores_to_labels(private_scores, thresholds)

metrics = {
    'method': 'deberta_v3_finetune',
    'model': MODEL_NAME,
    'folds': FOLDS,
    'seeds': SEEDS,
    'epochs': EPOCHS,
    'max_len': MAX_LEN,
    'batch_train': BATCH_TRAIN,
    'grad_accum_steps': GRAD_ACCUM_STEPS,
    'lr_encoder': LR_ENCODER,
    'lr_head': LR_HEAD,
    'oof_qwk': float(oof_qwk),
    'oof_mae': float(mean_absolute_error(y_class, oof_pred)),
    'oof_macro_f1': float(f1_score(y_class, oof_pred, average='macro')),
    'thresholds': [float(v) for v in thresholds],
    'label_distribution_combined': {int(k): int(v) for k, v in pd.Series(
        np.concatenate([public_pred, private_pred])).value_counts().sort_index().items()},
    'label_distribution_public': {int(k): int(v) for k, v in pd.Series(public_pred).value_counts().sort_index().items()},
    'label_distribution_private': {int(k): int(v) for k, v in pd.Series(private_pred).value_counts().sort_index().items()},
    'fold_log': fold_log,
}
(RUN_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))

pd.DataFrame({'id': train_full['id'], 'Label': y_class,
              'oof_score': oof_scores, 'oof_pred': oof_pred}).to_csv(
    RUN_DIR / 'oof_scores.csv', index=False)
pd.DataFrame({'id': public_full['id'], 'score': public_scores,
              'pred': public_pred}).to_csv(RUN_DIR / 'public_scores.csv', index=False)
pd.DataFrame({'id': private_full['id'], 'score': private_scores,
              'pred': private_pred}).to_csv(RUN_DIR / 'private_scores.csv', index=False)

combo = pd.concat([
    pd.DataFrame({'id': public_full['id'], 'Label': public_pred}),
    pd.DataFrame({'id': private_full['id'], 'Label': private_pred}),
], ignore_index=True)
submission = sample[['id']].merge(combo, on='id', how='left')
submission['Label'] = submission['Label'].astype(int)
submission.to_csv(RUN_DIR / 'deberta_v3_finetune_submission.csv', index=False)
print('submission rows =', len(submission))

## 10. Zip + download

In [ ]:
import pathlib, zipfile

# Platform-agnostic packaging.
# Always writes the zip next to RUN_DIR; only triggers Colab's download widget
# when actually running on Colab.
zip_path = pathlib.Path(OUT) / 'deberta_v3_finetune_outputs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in pathlib.Path(RUN_DIR).iterdir():
        zf.write(p, arcname=f'deberta_v3_finetune/{p.name}')
print('zipped:', zip_path, 'size MB =', round(zip_path.stat().st_size / 1e6, 2))

if PLATFORM == 'colab':
    from google.colab import files
    files.download(str(zip_path))
elif PLATFORM == 'kaggle':
    # /kaggle/working is the canonical "outputs" location; copy the zip there
    # so it appears in the kernel's Output tab.
    import shutil
    kaggle_out = pathlib.Path('/kaggle/working') / zip_path.name
    if zip_path.resolve() != kaggle_out.resolve():
        shutil.copy(zip_path, kaggle_out)
    print('available at:', kaggle_out)
else:
    print('local run — zip is at:', zip_path)